In [19]:
# Bloco 1: Instalação de dependências e Configuração do Ambiente

# Instalação das bibliotecas necessárias
!pip install -q --upgrade diffusers transformers accelerate peft datasets torch torchvision torchaudio torchao>=0.6.0


import torch
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# 1. Autenticação no Hugging Face

try:
    user_secrets = UserSecretsClient()
    token = user_secrets.get_secret("hugging-face")
    login(token)
    print("Autenticação no Hugging Face realizada com sucesso!")
except Exception as e:
    print("Erro na autenticação. Verifique seu Secret no Kaggle ou insira o token manualmente.")

# 2. Configuração de dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo de processamento: {device}")

Autenticação no Hugging Face realizada com sucesso!
Dispositivo de processamento: cuda


In [21]:
# Instala o edge-tts silenciosamente
!pip install edge-tts -q

from diffusers import StableDiffusionPipeline
from transformers import pipeline
import torch
import edge_tts
import asyncio

print("Carregando modelos... (isso pode levar alguns minutos)")

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Carregar Stable Diffusion com seu LoRA
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
lora_id = "homerio/estilo_arquitetura_modernista_brasilia_v4"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id, 
    torch_dtype=torch.float16
).to(device)

pipe.load_lora_weights(lora_id)
print("LoRA e Stable Diffusion carregados com sucesso!")

# 2. Carregar LLM para expansão de prompt (Llama 3.2 1B Instruct)
expansor = pipeline(
    "text-generation", 
    model="meta-llama/Llama-3.2-1B-Instruct", 
    device=device,
    torch_dtype=torch.float16 # Adicionado para economizar memória na GPU
)
print("Llama-3.2-1B-Instruct carregado com sucesso!")

print("Todos os modelos estão prontos!")

Carregando modelos... (isso pode levar alguns minutos)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


...

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Llama-3.2-1B-Instruct carregado com sucesso!
Todos os modelos estão prontos!


In [22]:
import gradio as gr
import tempfile
import edge_tts 
import asyncio

# Transformamos a função em 'async' por causa do edge_tts
async def pipeline_multimodal(tema):
    
    # 1. LLM expande o tema
    # Instrução clara para o Qwen gerar uma frase coesa
    instrucao = (f"Crie uma descrição visual detalhada para {tema}, em apenas uma frase, "
                 "DEVE ser ambientada estritamente em Brasília,"
                 "inicie a resposta exatamente com: 'estilo_arquitetura_modernista_brasilia'")
    
    # Gerar prompt
    resultado_llm = expansor(
        instrucao, 
        max_new_tokens=80, 
        do_sample=True, 
        temperature=0.5,
        clean_up_tokenization_spaces=False
    )
    
    texto_gerado = resultado_llm[0]["generated_text"]
      
    # 2. LIMPEZA DINÂMICA
    # Procuramos onde a nossa palavra-chave começa para ignorar o "bate-papo" do Qwen
    keyword = "estilo_arquitetura_modernista_brasilia"
    
    # Como o Qwen repete a instrução no output, vamos usar o rfind() 
    # para pegar a ÚLTIMA vez que a palavra aparece (que é a resposta dele)
    posicao = texto_gerado.rfind(keyword)
    
    if posicao != -1:
        prompt_final = texto_gerado[posicao:].strip()
    else:
        # Fallback: caso o modelo "esqueça" a instrução
        prompt_final = f"{keyword}, {tema}"
    
    # 3. Stable Diffusion gera a imagem
    imagem = pipe(
        prompt_final,  
        negative_prompt="desfocado, deformado, sobreposição de elementos, elementos desconexos, texto, marca d'água", 
        num_inference_steps=30, 
        guidance_scale=10
    ).images[0]
    
        # 4. TTS (Edge-TTS) gera o áudio
    caminho_audio = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3").name
    
    # Voz em português do Antonio
    communicate = edge_tts.Communicate(prompt_final, "pt-BR-AntonioNeural")
    await communicate.save(caminho_audio)
    
    return prompt_final, imagem, caminho_audio

# 5. Interface Gradio
demo = gr.Interface(
    fn=pipeline_multimodal,
    inputs=gr.Textbox(label="Tema", placeholder="Ex: Congresso Nacional ao pôr do sol"),
    outputs=[
        gr.Textbox(label="Prompt Expandido pelo Qwen"),
        gr.Image(label="Imagem Gerada"),
        gr.Audio(label="Descrição em Áudio")
    ],
    title="Ateliê Generativo - Arquitetura de Brasília",
    description="Digite um tema. O Qwen criará uma descrição detalhada e o Stable Diffusion irá gerar a imagem baseada nela."
)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7868
* Running on public URL: https://33f2f58910b68ddd8e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (111 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['terizada por uma forma retangular com duas torres laterais que se encontram em uma altura de 6 2 metros , formando um " x "']


  0%|          | 0/30 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['u de brasília é um dos principais monumentos arquitetônicos do brasil , projetado pelo arquiteto oscar niemeyer e inspirado na']


  0%|          | 0/30 [00:00<?, ?it/s]

...

  0%|          | 0/30 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['é coberto por telhados de arame farpado e revestido de azulejos . a f']


  0%|          | 0/30 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['ito e aço , se ergue como uma estrela brilhante no cenário urbano de brasília , com uma']


  0%|          | 0/30 [00:00<?, ?it/s]